In [ ]:
!pip install prophet google-cloud-bigquery pandas db-dtypes -q


In [ ]:
import pandas as pd
import numpy as np

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

client = bigquery.Client(project="retail-forecasting-506113")

print("BigQuery connected successfully")

BigQuery connected successfully


In [ ]:
query = """
SELECT
    item_id,
    store_id,
    ds,
    y
FROM
    `retail-forecasting-506113.retail_forecasting.prophet_high_volume`
ORDER BY
    item_id,
    store_id,
    ds
"""

df = client.query(query).to_dataframe()

print("Rows:", len(df))
df.head()

Rows: 9705


,item_id,store_id,ds,y
0,FOODS_3_090,CA_1,2011-01-29,107.0
1,FOODS_3_090,CA_1,2011-01-30,182.0
2,FOODS_3_090,CA_1,2011-01-31,47.0
3,FOODS_3_090,CA_1,2011-02-01,47.0
4,FOODS_3_090,CA_1,2011-02-02,62.0


In [ ]:
holiday_query = """
SELECT
    ds,
    holiday
FROM
    `retail-forecasting-506113.retail_forecasting.prophet_holidays`
ORDER BY
    ds
"""

holidays = client.query(holiday_query).to_dataframe()

print("Holiday rows:", len(holidays))
holidays.head()

Holiday rows: 167


,ds,holiday
0,2011-02-06,SuperBowl
1,2011-02-14,ValentinesDay
2,2011-02-21,PresidentsDay
3,2011-03-09,LentStart
4,2011-03-16,LentWeek2


In [ ]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Make sure dates are correctly formatted
df["ds"] = pd.to_datetime(df["ds"])
df["y"] = pd.to_numeric(df["y"])

# Store results
model_results = []
forecast_results = []

# Train Prophet separately for each item/store
for (item, store), group in df.groupby(["item_id", "store_id"]):

    group = group[["ds", "y"]].sort_values("ds").reset_index(drop=True)

    # Last 28 days for testing
    train = group.iloc[:-28].copy()
    test = group.iloc[-28:].copy()

    print(f"Training: {item} - {store}")
    print(f"Train rows: {len(train)}, Test rows: {len(test)}")

    # Prophet model
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        holidays=holidays
    )

    # Train model
    model.fit(train)

    # Predict test period
    future = test[["ds"]].copy()
    forecast = model.predict(future)

    # Actual and predicted values
    actual = test["y"].values
    predicted = forecast["yhat"].values

    # Metrics
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    model_results.append({
        "item_id": item,
        "store_id": store,
        "MAE": mae,
        "RMSE": rmse
    })

    # Save forecast results
    result = test.copy()
    result["item_id"] = item
    result["store_id"] = store
    result["yhat"] = predicted
    result["yhat_lower"] = forecast["yhat_lower"].values
    result["yhat_upper"] = forecast["yhat_upper"].values

    forecast_results.append(result)

    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print("-" * 40)

# Combine results
metrics_df = pd.DataFrame(model_results)
prophet_forecasts = pd.concat(forecast_results, ignore_index=True)

print("Prophet training completed.")

Training: FOODS_3_090 - CA_1
Train rows: 1913, Test rows: 28
MAE: 30.36
RMSE: 34.32
----------------------------------------
Training: FOODS_3_090 - CA_3
Train rows: 1913, Test rows: 28
MAE: 58.91
RMSE: 64.22
----------------------------------------
Training: FOODS_3_586 - CA_3
Train rows: 1913, Test rows: 28
MAE: 7.77
RMSE: 10.17
----------------------------------------
Training: FOODS_3_586 - TX_2
Train rows: 1913, Test rows: 28
MAE: 13.21
RMSE: 16.68
----------------------------------------
Training: FOODS_3_586 - TX_3
Train rows: 1913, Test rows: 28
MAE: 14.46
RMSE: 18.15
----------------------------------------
Prophet training completed.


In [ ]:
prophet_forecasts.head(10)

,ds,y,item_id,store_id,yhat,yhat_lower,yhat_upper
0,2016-04-25,48.0,FOODS_3_090,CA_1,30.948491,-28.187799,90.137132
1,2016-04-26,35.0,FOODS_3_090,CA_1,28.189661,-30.157310,85.057552
2,2016-04-27,34.0,FOODS_3_090,CA_1,29.010209,-31.094277,86.387878
3,2016-04-28,67.0,FOODS_3_090,CA_1,30.688685,-26.024936,89.222161
4,2016-04-29,63.0,FOODS_3_090,CA_1,50.856011,-5.666608,107.297131
5,2016-04-30,99.0,FOODS_3_090,CA_1,80.479582,24.823428,135.351579
6,2016-05-01,71.0,FOODS_3_090,CA_1,46.431462,-13.242121,100.091127
7,2016-05-02,59.0,FOODS_3_090,CA_1,27.905711,-27.722308,84.022261
8,2016-05-03,35.0,FOODS_3_090,CA_1,24.078036,-30.896223,83.418865
9,2016-05-04,47.0,FOODS_3_090,CA_1,23.801268,-31.417226,76.782298


In [ ]:
print("Forecast rows:", len(prophet_forecasts))
print("Columns:")
print(prophet_forecasts.columns.tolist())

Forecast rows: 140
Columns:
['ds', 'y', 'item_id', 'store_id', 'yhat', 'yhat_lower', 'yhat_upper']
